## Lab 1. MLP

In [23]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


#daecarga
houses = fetch_california_housing(as_frame=True)
df = houses.frame

### 1. Dataset

In [19]:
print("=== EXPLORACION Y PREPARACION DATASET ===")
print("\nDimensiones: ")
print(f" - Filas: {df.shape[0]}")
print(f" - Columnas: {df.shape[1]}")
print(f"\n{df.info()}")
print(f"\nEstadisticas:\n{df.describe()}")
print(f"\nVariable objetivo: {houses.target_names[0]}")
print(f"Variable feature: {list(houses.feature_names)}")

null_counts = df.isnull().sum()
print(f"\nValores nulos: {null_counts[null_counts > 0].to_dict() if any(null_counts > 0) else 'No hay valores nulos'}")
duplicates = df.duplicated().sum()
print(f"Valores duplicados: {duplicates}")

#outliers/atipicos con IQR
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]

    return {
        'col': col,
        'lower': lower,
        'upper': upper,
        'n_outliers': len(outliers),
        'total': len(outliers),
        'pct': len(outliers) / len(df) * 100
    }

print("Valores atipicos/variable:")
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        info = detect_outliers_iqr(df, col)
        if info['n_outliers'] > 0:
            print(f"   - {col}: {info['n_outliers']} outliers ({info['pct']:.1f}%)")
        else:
            print(f"   - {col}: No outliers")




=== EXPLORACION Y PREPARACION DATASET ===

Dimensiones: 
 - Filas: 20640
 - Columnas: 9
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB

None

Estadisticas:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.

### 2. Exploracion y preparación de datos

In [ ]:
def tratamiento_outlier(df, column):
    info = detect_outliers_iqr(df, column)
    n_outliers = info['total']
    pct_outliers = info['pct']
    
    print(f"\n--- {column} ---")
    print(f"Outliers: {n_outliers} ({pct_outliers:.1f}%)")
    print(f"Limite inferior: {info['lower']:.3f}")
    print(f"Limite superior: {info['upper']:.3f}")
    
    #valores extremos
    min_val = df[column].min()
    max_val = df[column].max()

    print(f"Rango: [{min_val:.2f}, {max_val:.2f}]")
    if pct_outliers == 0:
        rec = "No hay outliers detectados"
    elif pct_outliers < 1:
        rec = "Eliminar (pocos outliers, <1% del total)"
    elif pct_outliers < 3:
        rec = "Evaluar según modelo (1-3% del total)"
    elif pct_outliers < 5:
        rec = "Mantener o winsorizar (3-5% del total)"
    else:
        rec = "Mantener (más del 5%, es parte de la distribución)"

    return rec

for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        tratamiento_outlier(df, col)

outliers_info = []
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        info = detect_outliers_iqr(df, col)
        outliers_info.append(info)

#prom de %
pct_outliers_total = np.mean([info['pct'] for info in outliers_info])
print(f"Porcentaje promedio de outliers: {pct_outliers_total:.1f}%")

if pct_outliers_total < 1:
    decision = "Eliminar atipicos (son pocos y no afectan significativamente)"
elif pct_outliers_total < 3:
    decision = "Mantener"
elif pct_outliers_total < 5:
    decision = "Mantener atipicos (son parte de la distribución real)"
else:
    decision = "Mantener atipicos (mas del 5% es que los datos pueden llegar a eso)"

print(f"\nDecision final: {decision}")


#escalar y separar datos 
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']
# 70 entrenamiento, 15 validación, 15 prueba
X_train, X_temp, y_train, y_temp = train_test_split(X,y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = RobustScaler()

print("Escalando datos...")

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Scaler ajustado, {len(X_train)} muestras")
print(f"Validacion ({len(X_val)}) y prueba ({len(X_test)}) transformadas")



--- MedInc ---
Outliers: 681 (3.3%)
Limite inferior: -0.706
Limite superior: 8.013
Rango: [0.50, 15.00]

--- HouseAge ---
Outliers: 0 (0.0%)
Limite inferior: -10.500
Limite superior: 65.500
Rango: [1.00, 52.00]

--- AveRooms ---
Outliers: 511 (2.5%)
Limite inferior: 2.023
Limite superior: 8.470
Rango: [0.85, 141.91]

--- AveBedrms ---
Outliers: 1424 (6.9%)
Limite inferior: 0.866
Limite superior: 1.240
Rango: [0.33, 34.07]

--- Population ---
Outliers: 1196 (5.8%)
Limite inferior: -620.000
Limite superior: 3132.000
Rango: [3.00, 35682.00]

--- AveOccup ---
Outliers: 711 (3.4%)
Limite inferior: 1.151
Limite superior: 4.561
Rango: [0.69, 1243.33]

--- Latitude ---
Outliers: 0 (0.0%)
Limite inferior: 28.260
Limite superior: 43.380
Rango: [32.54, 41.95]

--- Longitude ---
Outliers: 0 (0.0%)
Limite inferior: -127.485
Limite superior: -112.325
Rango: [-124.35, -114.31]
Porcentaje promedio de outliers: 2.7%

Decision final: Mantener
Division dataset:
	 - Entrenamiento: 14448 muestras
	 - Vali

**¿Cuántas observaciones y cuántas variables tiene el dataset?** Total de servaciones y 9 variables\
**¿Qué representa cada variable (feature) y cuál es la variable objetivo (target)?**
La variable objetivo es MedHouseVal\
 Fatures:   
 - Longitude: que tan al oeste esta la casa (mientras mas alto el valor más al oeste)
 - Latitude: que tan al norte esta la casa (mientras mas alto el valor más al norte)
 - HouseAge: Edad de una casa en la cuadra
 - AveRooms: promedio de cuartos en la cuadra
 - AveBedrms: promedio de habitaciones en la cuadra
 - Population: total de poblacion en la cuadra
 - AveOccup: promedio de personas por casa
 - MedInc: Media de ingresos por casa por cuadra

**¿Hay valores nulos, duplicados o atípicos (outliers)? ¿Cómo los trató?**

Investigué un poco sobre esto y encontré que si eran menos del 5% total, se podian mantener

**¿Qué variables son numéricas y cuáles categóricas? ¿Cómo codificó las categóricas?**
- Numéricas: Las 9 variables son numéricas, especificamente float64
- Categóricas: Ninguna

**¿Fue necesario normalizar o escalar las variables numéricas?**
Debido a la gran diferencia de algunos rangos de valores, si, debido a que hay outliers robuscaler es mejor opcion ya uqe este es menos sensible a estos

### 3. Investigación: optimizadores y capas de PyTorch para el MLP

- nn. Linear: Aplica transformacion linear afin a datos de entrada  $y=xA^T+b$
- nn.ReLU: Aplica la funcion de unidad lineal rectificada por elemento  $ReLU(x) = (x)^+ = max(0,x)$
- nn.LeakyReLU: Aplica la funcion LeakyReLU por elemento  $LeakuReLU = max(0,x)+negative_slope∗min(0,x)$ o $\begin{cases} x &\text{if } x \ge 0 \\ negative Slope * X \text{, otherwise} \end{cases}$
- nn.Tanh: Aplica Tangente Hiperbólica por elemento  $Tanh(x)= \frac{exp(x)−exp(−x)}{exp(x)+exp(−x)}$
- nn.Dropout: Al entrenar algunos elemetos del input tensor se covnierten a 0 con proabilidad p. Estos elementos son elegidos de forma aleatoria e independiente de cada forward call.
- nn.BatchNorm1d: aplica la normalizacion por lotes a datos que son de una dimension (como vectores de series temporales)
- nn.MSELoss: Calcula error cuadrático (MSE) entre los valores reales y los que se predicen
- nn.L1Loss: Calcula el MAE (error absoluto medio) entre la salida y el valor real objetivo.
- nn.SmoothL1Loss: Funcion para calcular la perdida en aprendizaje automatico que combina las ventajas de L1 y L2 (MSE) usando un termino cuadratico para errores pequeños y lineal para grandes.
- torch.optim:
    - SGD: Implementa descenso de gradiente estocástico, actualiza pesos y sesgos de las capas al restar el gradiente a la funcion de perdida y multiplicando con la tasa de aprendizaje, con eso minimiza el error del modelo. La tasa de aprendizaje es fija para todos los pesos, no consume tanta memoria y deja añadir parametros opcionales. El parámetro lr controla el tamaño del paso del optimizador al actualizar pesos, weight_decay aplica una penalización con el fin de evitar un sobreajuste que pueda afectar los pesos.
    - Adam: Implementa el algoritmo Adam (Adaptive Moment Estimation), este es un algoritmo de optimización para entrenar modelos de Deep learning, combina el momento con tasas de aprendizaje adaptativas por cada parámetro con el fin de obtener una convergencia rápida y estable sin necesitar de ajustes complejos en los hiperparámetros. Ajusta los pesos y sesgos de cada capa de forma independiente después del cálculo de retropropagación, estabiliza el aprendizaje al usar promedios móviles exponenciales de los gradientes y puede funcionar de forma bastante efectiva con la tasa base.
    - RMSprop: Ajusta la tasa de aprendizaje para cada parámetro de forma adaptativa haciendo uso de los gradientes, hace los cálculos para determinar un promedio ponderado del cuadrado de los gradientes más recientes con el fin de escalar las actualizaciones de pesos y sesgos de cada capa. La tasa de aprendizaje se va adaptando a cada uno de los parámetros de forma individual, la inclusión de momento es opcional y permite que el gradiente se normalice por medio de una estimación de varianza. 


### 4. Entrenamiento e iteración de hiperparámetros

In [ ]:
print("Division dataset:")
print(f"\t - Entrenamiento: {len(X_train)} muestras")
print(f"\t - Validacion: {len(X_val)} muestras")
print(f"\t - Prueba: {len(X_test)} muestras")

#para pythorch con tensor
X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
X_val_tensor = torch.FloatTensor(X_val_scaled)

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1,1)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1,1)
y_val_tensor = torch.FloatTensor(y_val.values).reshape(-1, 1)

#dataloaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

#MLP
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, activation='relu', dropout=0.0):
        super(MLP, self).__init__()

        layers = []
        prev_dim = input_dim

        #capas ocultas
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim)) #normalzacion ayduda con conver.

            #activ
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.1))
            elif activation == 'selu':
                layers.append(nn.SELU())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'sigmoid':
                layers.append(nn.Sigmoid())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev_dim = hidden_dim

        #capa salida/regresion
        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def training(model, train_loader, val_loader, optimizer, criterion, epochs, device='cpu'):
    model.to(device)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        #training
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)

        train_loss/= len(train_loader.dataset)
        train_losses.append(train_loss)

        #val
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
    
    return train_losses, val_losses

def evaluation_model(model, X_tensor, y_tensor, device='cpu'):
    model.eval()
    with torch.no_grad():
        X_tensor = X_tensor.to(device)
        predictions = model(X_tensor).cpu().numpy()
        y_true = y_tensor.numpy()

    mse = mean_squared_error(y_true, predictions)
    mae = mean_absolute_error(y_true, predictions)
    rmse = np.sqrt(mse)

    return {
        'MSE': mse,
        'MAE': mae,
        'RMSE': rmse,
        'Predictions': predictions
    }

print("=== 10 Iteraciones de modelo ===")
#config_2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
#perdida
criterion = nn.MSELoss()

result = []
best_val_rmse = float('inf')
best_model = None
best_config = None

#modelo base
print("\n\t1. Modelo Base")
config = {
    'hidden_layers': [64, 32],
    'activation': 'relu',
    'optimizer': 'adam',
    'learning_rate': 0.001,
    'batch_size': 64,
    'epochs': 100,
    'dropout': 0.0,
    'weight_decay': 0.0
}
print("\tConfiguracion:")
for key, value in config.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config['hidden_layers'],
    activation=config['activation'],
    dropout=config['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
#optm
if config['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
elif config['optimizer'] == 'sgd':
    optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'], momentum=0.9)
elif config['optimizer'] == 'rmsprop':
    optimizer = optim.RMSprop(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 1,
    'config': config,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\nResultados validacion:")
print(f"- MSE: {metrics['MSE']:.4f}")
print(f"- MAE: {metrics['MAE']:.4f}")
print(f"- RMSE: {metrics['RMSE']:.4f}")

#modelo arquitectura
print("\n\t2. Modelo con cambios en Arquitectura")

config_2 = config.copy()

config_2['hidden_layers'] = [128, 64, 32]
config_2['epochs'] = 80

print("\tConfiguracion:")
for key, value in config_2.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_2['hidden_layers'],
    activation=config_2['activation'],
    dropout=config_2['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_2['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_2['batch_size'], shuffle=False)
#optm
if config_2['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_2['learning_rate'], weight_decay=config_2['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_2['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 2,
    'config': config_2,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


#modelo fucinones de activacion
print("\n\t3. Modelo con cambios en Funcion de Activacion")

config_3 = config.copy()

config_3['activation'] = 'leaky_relu'
config_3['hidden_layers'] = [64,32]

print("\tConfiguracion:")
for key, value in config_3.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_3['hidden_layers'],
    activation=config_3['activation'],
    dropout=config_3['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_3['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_3['batch_size'], shuffle=False)
#optm
if config_3['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_3['learning_rate'], weight_decay=config_3['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_3['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 3,
    'config': config_3,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


#modelo optimizador y learninr rate
print("\n\t4. Modelo con cambios en Optimizador y Learning Rate")

config_4 = config.copy()

config_4['optimizer'] = 'sgd'
config_4['learning_rate'] = 0.01

print("\tConfiguracion:")
for key, value in config_4.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_4['hidden_layers'],
    activation=config_4['activation'],
    dropout=config_4['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_4['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_4['batch_size'], shuffle=False)
#optm
if config_4['optimizer'] == 'sgd':
    optimizer = optim.SGD(model.parameters(), lr=config_4['learning_rate'], weight_decay=config_4['weight_decay'], momentum=0.9)

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_4['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 4,
    'config': config_4,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


#modelos con cmabios de regularizacion
print("\n\t5. Modelo con cambios en Regularizacion: L2")

config_5 = config.copy()

config_5['weight_decay'] = 0.001

print("\tConfiguracion:")
for key, value in config_5.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_5['hidden_layers'],
    activation=config_5['activation'],
    dropout=config_5['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_5['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_5['batch_size'], shuffle=False)
#optm
if config_5['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_5['learning_rate'], weight_decay=config_5['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_5['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 5,
    'config': config_5,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


print("\n\t6. Modelo con cambios en Regularizacion: Dropout")

config_6 = config.copy()
config_6['dropout'] = 0.2

print("\tConfiguracion:")
for key, value in config_6.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_6['hidden_layers'],
    activation=config_6['activation'],
    dropout=config_6['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_6['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_6['batch_size'], shuffle=False)
#optm
if config_6['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_6['learning_rate'], weight_decay=config_6['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_6['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 6,
    'config': config_6,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


print("\n\t7. Modelo con cambios en Regularizacion: Dropout y L2")

config_7 = config.copy()

config_7['dropout'] = 0.3
config_7['weight_decay'] = 0.001

print("\tConfiguracion:")
for key, value in config_3.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_7['hidden_layers'],
    activation=config_7['activation'],
    dropout=config_7['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_7['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_7['batch_size'], shuffle=False)
#optm
if config_7['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_7['learning_rate'], weight_decay=config_7['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_7['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 7,
    'config': config_7,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


#modelos bach size y no. epochs
print("\n\t8. Modelo con cambios en Batch size y epoch")

config_8 = config.copy()

config_8['batch_size'] = 32
config_8['epochs']=120

print("\tConfiguracion:")
for key, value in config_3.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_8['hidden_layers'],
    activation=config_8['activation'],
    dropout=config_8['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_8['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_8['batch_size'], shuffle=False)
#optm
if config_8['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_8['learning_rate'], weight_decay=config_8['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_8['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 8,
    'config': config_8,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


print("\n\t9. Modelo con cambios en Learning Rate: 0.0005")

config_9 = config.copy()

config_9['learning_rate']= 0.0005
config_9['epochs'] = 120

print("\tConfiguracion:")
for key, value in config_9.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_9['hidden_layers'],
    activation=config_9['activation'],
    dropout=config_9['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_9['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_9['batch_size'], shuffle=False)
#optm
if config_9['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_9['learning_rate'], weight_decay=config_9['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_9['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 9,
    'config': config_9,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")


#Combinacion
print("\n\t10. Modelo con combinacion más exitosa")

config_10 = {
    'hidden_layers': [128, 64, 32],
    'activation': 'leaky_relu',
    'optimizer': 'adam',
    'learning_rate': 0.0005,
    'batch_size': 32,
    'epochs': 100,
    'dropout': 0.2,
    'weight_decay': 0.001
}

print("\tConfiguracion:")
for key, value in config_10.items():
    print(f"\t  {key}: {value}")

#crear modelos
model = MLP(
    input_dim=X_train_tensor.shape[1],
    hidden_layers=config_10['hidden_layers'],
    activation=config_10['activation'],
    dropout=config_10['dropout']
)
#dataloader
train_loader = DataLoader(train_dataset, batch_size=config_10['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config_10['batch_size'], shuffle=False)
#optm
if config_10['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=config_10['learning_rate'], weight_decay=config_10['weight_decay'])

#entrenar
train_losses, val_losses = training(model, train_loader, val_loader, optimizer, criterion, epochs=config_10['epochs'], device=device)

#eval
metrics = evaluation_model(model, X_val_tensor, y_val_tensor, device)
metrics['train_loss'] = train_losses[-1]
metrics['val_loss'] = val_losses[-1]

result.append({
    'iteration': 10,
    'config': config_10,
    'metrics': metrics,
    'train_losses': train_losses,
    'val_losses': val_losses
})

print(f"\n\tResultados validacion:")
print(f"\t- MSE: {metrics['MSE']:.4f}")
print(f"\t- MAE: {metrics['MAE']:.4f}")
print(f"\t- RMSE: {metrics['RMSE']:.4f}")



#resultados
print("=== TABLA DE RESULTADOS ===")
print("-"*120)
print(f"{'Iter':^5} | {'Arquitectura':^25} | {'Activación':^10} | {'Optimizador':^10} | {'LR':^8} | {'Dropout':^7} | {'L2':^6} | {'RMSE':^8} | {'MAE':^8}")
print("-"*120)

for r in result:
    config = r['config']
    arch = str(config['hidden_layers'])
    print(f"{r['iteration']:^5} | {arch:^25} | {config['activation']:^10} | {config['optimizer']:^10} | {config['learning_rate']:^8.4f} | {config['dropout']:^7.1f} | {config['weight_decay']:^6.4f} | {r['metrics']['RMSE']:^8.4f} | {r['metrics']['MAE']:^8.4f}")

print("-"*120)

#encontrar la mejor configuracion
best_idx = np.argmin([r['metrics']['RMSE'] for r in result])
best_result = result[best_idx]
